This notebook functions as the trial workspace for identifying skill gap between the job-seeker and real required skillset or job tasks.

In [1]:
# import shutil
# import os

# db_path = "./chroma_db/dynamic_csv"

# if os.path.exists(db_path):
#     shutil.rmtree(db_path)

In [2]:
!pip install -U -q \
  "google-genai" \
  langchain \
  langchain_community \
  langchain-core \
  langchain-google-genai \
  langchain_chroma \
  langchain-text-splitters \
  pypdf \
  chromadb \
  langchain-huggingface \
  sentence-transformers

In [1]:
# Import libraries
from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from google import genai
from langchain_chroma import Chroma
from dotenv import load_dotenv
import os
import json

from pdfminer.high_level import extract_text
import re
from langchain_core.messages import SystemMessage
from langchain_core.prompts import HumanMessagePromptTemplate, ChatPromptTemplate, PromptTemplate

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_39220\2026199240.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader
c:\Users\Lenovo\anaconda3\envs\final_project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Fetch and print models that support text/content generation
print("Available Gemini Models:\n")
for m in client.models.list():
    for action in m.supported_actions:
        if action == "generateContent":
            print(f"Model ID: {m.name}")
            print(f"Display Name: {m.display_name}")
            print("-" * 30)

Available Gemini Models:

Model ID: models/gemini-2.5-flash
Display Name: Gemini 2.5 Flash
------------------------------
Model ID: models/gemini-2.5-pro
Display Name: Gemini 2.5 Pro
------------------------------
Model ID: models/gemini-2.0-flash
Display Name: Gemini 2.0 Flash
------------------------------
Model ID: models/gemini-2.0-flash-001
Display Name: Gemini 2.0 Flash 001
------------------------------
Model ID: models/gemini-2.0-flash-lite-001
Display Name: Gemini 2.0 Flash-Lite 001
------------------------------
Model ID: models/gemini-2.0-flash-lite
Display Name: Gemini 2.0 Flash-Lite
------------------------------
Model ID: models/gemini-2.5-flash-preview-tts
Display Name: Gemini 2.5 Flash Preview TTS
------------------------------
Model ID: models/gemini-2.5-pro-preview-tts
Display Name: Gemini 2.5 Pro Preview TTS
------------------------------
Model ID: models/gemma-4-26b-a4b-it
Display Name: Gemma 4 26B A4B IT
------------------------------
Model ID: models/gemma-4-31b-i

Available Gemini Models:

Model ID: models/gemini-2.5-flash
Display Name: Gemini 2.5 Flash
------------------------------
Model ID: models/gemini-2.5-pro
Display Name: Gemini 2.5 Pro
------------------------------
Model ID: models/gemini-2.0-flash
Display Name: Gemini 2.0 Flash
------------------------------
Model ID: models/gemini-2.0-flash-001
Display Name: Gemini 2.0 Flash 001
------------------------------
Model ID: models/gemini-2.0-flash-lite-001
Display Name: Gemini 2.0 Flash-Lite 001
------------------------------
Model ID: models/gemini-2.0-flash-lite
Display Name: Gemini 2.0 Flash-Lite
------------------------------
Model ID: models/gemini-2.5-flash-preview-tts
Display Name: Gemini 2.5 Flash Preview TTS
------------------------------
Model ID: models/gemini-2.5-pro-preview-tts
Display Name: Gemini 2.5 Pro Preview TTS
------------------------------
Model ID: models/gemma-4-26b-a4b-it
Display Name: Gemma 4 26B A4B IT
------------------------------
Model ID: models/gemma-4-31b-it
Display Name: Gemma 4 31B IT
------------------------------
Model ID: models/gemini-flash-latest
Display Name: Gemini Flash Latest
------------------------------
Model ID: models/gemini-flash-lite-latest
Display Name: Gemini Flash-Lite Latest
------------------------------
Model ID: models/gemini-pro-latest
Display Name: Gemini Pro Latest
------------------------------
Model ID: models/gemini-2.5-flash-lite
Display Name: Gemini 2.5 Flash-Lite
------------------------------
Model ID: models/gemini-2.5-flash-image
Display Name: Nano Banana
------------------------------
Model ID: models/gemini-3-pro-preview
Display Name: Gemini 3 Pro Preview
------------------------------
Model ID: models/gemini-3-flash-preview
Display Name: Gemini 3 Flash Preview
------------------------------
Model ID: models/gemini-3.1-pro-preview
Display Name: Gemini 3.1 Pro Preview
------------------------------
Model ID: models/gemini-3.1-pro-preview-customtools
Display Name: Gemini 3.1 Pro Preview Custom Tools
------------------------------
Model ID: models/gemini-3.1-flash-lite-preview
Display Name: Gemini 3.1 Flash Lite Preview
------------------------------
Model ID: models/gemini-3.1-flash-lite
Display Name: Gemini 3.1 Flash Lite
------------------------------
Model ID: models/gemini-3-pro-image-preview
Display Name: Nano Banana Pro
------------------------------
Model ID: models/gemini-3-pro-image
Display Name: Nano Banana Pro
------------------------------
Model ID: models/nano-banana-pro-preview
Display Name: Nano Banana Pro
------------------------------
Model ID: models/gemini-3.1-flash-image-preview
Display Name: Nano Banana 2
------------------------------
Model ID: models/gemini-3.1-flash-image
Display Name: Nano Banana 2
------------------------------
Model ID: models/gemini-3.1-flash-lite-image
Display Name: Nano Banana 2 Lite
------------------------------
Model ID: models/gemini-3.5-flash
Display Name: Gemini 3.5 Flash
------------------------------
Model ID: models/gemini-omni-flash-preview
Display Name: Gemini Omni Flash Preview
------------------------------
Model ID: models/lyria-3-clip-preview
Display Name: Lyria 3 Clip Preview
------------------------------
Model ID: models/lyria-3-pro-preview
Display Name: Lyria 3 Pro Preview
------------------------------
Model ID: models/gemini-3.1-flash-tts-preview
Display Name: Gemini 3.1 Flash TTS Preview
------------------------------
Model ID: models/gemini-robotics-er-1.5-preview
Display Name: Gemini Robotics-ER 1.5 Preview
------------------------------
Model ID: models/gemini-robotics-er-1.6-preview
Display Name: Gemini Robotics-ER 1.6 Preview
------------------------------
Model ID: models/gemini-2.5-computer-use-preview-10-2025
Display Name: Gemini 2.5 Computer Use Preview 10-2025
------------------------------
Model ID: models/antigravity-preview-05-2026
Display Name: Antigravity Agent Preview
------------------------------
Model ID: models/deep-research-max-preview-04-2026
Display Name: Deep Research Max Preview (Apr-21-2026)
------------------------------
Model ID: models/deep-research-preview-04-2026
Display Name: Deep Research Preview (Apr-21-2026)
------------------------------
Model ID: models/deep-research-pro-preview-12-2025
Display Name: Deep Research Pro Preview (Dec-12-2025)
------------------------------

In [7]:
# Load the `.env` file that contains secret variables like API Keys
load_dotenv()

# We set the Google API Key for authentication when using Google's AI models. This key is necessary to access the services and is kept secret to prevent unauthorized use.
# Ensure you keep the API Key secret in real projects to prevent others from stealing your quota.
os.environ['GEMINI_API_KEY'] = os.getenv('GEMINI_API_KEY')
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
# General variable
EMBEDDING_MODEL = "models/gemini-embedding-2-preview"
CHAT_MODEL = "gemini-3.1-flash-lite-preview"
EMBEDDING_MODEL_HF = "sentence-transformers/all-MiniLM-L6-v2"
SPLITTER = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=100)

In [8]:
# Embedding
## Using Gemini
embedding_model_gemini = GoogleGenerativeAIEmbeddings(
    google_api = GEMINI_API_KEY,
    model = EMBEDDING_MODEL
)

## Using Hugging Face (alternative)
embedding_model_hf = HuggingFaceEmbeddings(
  model_name=EMBEDDING_MODEL_HF,
  model_kwargs={'device': 'cpu'}, # Additional options to specify that the model should run on CPU instead of GPU.
  encode_kwargs={'normalize_embeddings': True} # Additional options to specify that the embeddings should be normalized (converted to unit vectors) after encoding.
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7928.60it/s]


Processing CSV files

In [2]:
def job_processing(file_path, meta_data_cols: list):
    """
    Load, clean, embed, and storing csv file to vector database.

    Arguments
        - file_path: CSV document of job listings
        - meta_data_cols: Column names that will be used for filtering 
    """
    loader_csv = CSVLoader(file_path, metadata_columns=meta_data_cols, csv_args={
        "delimiter": ";"
    })
    docs = loader_csv.load()
    all_contents = [post for post in docs]
    chunks = SPLITTER.split_documents(all_contents)
    
    db_dynamic = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model_hf,
        persist_directory="./chroma_db/dynamic_csv"
    )
    print("Stored to vector db using Hugging Face.")
        
    print(f"Dynamic index updated: {len(chunks)} chunks")

CV Processing

In [ ]:
chat_model = ChatGoogleGenerativeAI(
            google_api_key=GEMINI_API_KEY,
            model=CHAT_MODEL
        )

In [13]:
def extract_cv(cv_path: str) -> str:
    loader = extract_text(cv_path) # Load file

    # Clean file
    cv_text = loader.replace("\xa0", " ") # Remove non-breaking space
    cv_text = "\n".join(line.rstrip() for line in cv_text.splitlines()) # Remove trailing whitespace for each row
    cv_text = re.sub(r"\n{3,}", "\n\n", cv_text) # Two new lines maximum

    return cv_text

Gap analysis between extracted CV and job openings

In [14]:
# # Match pipeline
# def match_cv_to_jobs(cv_path: str, value_to_extract: str):
#     cv_profile = extract_cv(cv_path)
            
#     # Load index  
#     dynamic_store = Chroma(
#         persist_directory="./chroma_db/dynamic_csv",
#         embedding_function=embedding_model_hf
#     )

#     query = f"""
#         Position:
#         {value_to_extract}

#         Profile candidate:
#         {cv_profile}
#     """
#     # Find matching job
#     job_matches = dynamic_store.similarity_search(query=query, k=5)
#     jobs_text = "\n\n---\n\n".join([d.page_content for d in job_matches]) # Formatting
 
#     # LLM
#     chat_model = ChatGoogleGenerativeAI(
#             google_api_key=GEMINI_API_KEY,
#             model=CHAT_MODEL
#         )
    
#     prompt = PromptTemplate.from_template("""
#         You are an AI Career Advisor.
#         If necessary, you may add limited relevant information to maximize the user experience.
#         Include logical reasoning for all your answers.

#         CANDIDATE PROFILE:
#         {cv_profile}

#         JOB VACANCIES:
#         {job_listings}

#         Provide:
#         1. CV rating based on writing quality (in percentage).
#         2. CV rating based on relevance to the job opening (in percentage).
#         3. Top 5 most suitable job openings by position, name of company, and industry (if information is available).
#         4. Predict the average salary user will receive.
#         5. Rate the user's skills (in percentage) based on the 5 most crucial skills for the job vacancy.
#         6. Recommendations to close the skill gap and give a better chance of the user to be accepted in these job openings.
#     """)

#     chain = prompt | chat_model
#     response = chain.invoke({
#         "cv_profile": cv_profile,
#         "job_listings": jobs_text
#     })

#     return "\n".join(
#                 part["text"]
#                 for part in response.content
#                 if part["type"] == "text"
#             )

In [15]:
# Match pipeline
def match_cv_to_jobs(cv_path: str, value_to_extract: str):
    cv_profile = extract_cv(cv_path)
            
    # Load index  
    dynamic_store = Chroma(
        persist_directory="./chroma_db/dynamic_csv",
        embedding_function=embedding_model_hf
    )

    query = f"""
        Position:
        {value_to_extract}

        Profile candidate:
        {cv_profile}
    """
    # Find matching job
    job_matches = dynamic_store.similarity_search(query=query, k=5)
    jobs_text = "\n\n---\n\n".join([d.page_content for d in job_matches]) # Formatting
 
    # LLM
    chat_model = ChatGoogleGenerativeAI(
            google_api_key=GEMINI_API_KEY,
            model=CHAT_MODEL
        )
    
    prompt = PromptTemplate.from_template("""
        You are an AI Career Advisor specializing in resume evaluation and job matching.

        Your task is to analyze the candidate's CV against the retrieved job vacancies.

        Instructions:
        - Return ONLY a valid JSON object.
        - Do NOT wrap the response inside markdown or 'json'.
        - Do NOT add any explanation before or after the JSON.
        - Follow the schema exactly.
        - Do NOT omit any key.
        - If information is unavailable, use null.
        - Integer fields must contain only numbers.
        - Percentage values must be integers between 0 and 100.
        - Arrays must always exist, even if empty.

        Candidate Profile:
        {cv_profile}

        Retrieved Job Vacancies:
        {job_listings}

        Return this schema exactly:

        {{
            "candidate": {{
                "name": string,
                "target_role": {value_to_extract},
                "professional_summary": string
            }},

            "scores": {{
                "cv_quality": integer (1-100),
                "job_relevance": integer (1-100),
                "overall_readiness": integer (1-100)
            }},

            "salary_prediction": {{
                "currency": string,
                "average": integer,
                "minimum": integer,
                "maximum": integer
            }},

            "top_jobs": [
                {{
                    "rank": integer,
                    "job_title": string,
                    "company": string,
                    "industry": string,
                    "location": string,
                    "match_score": integer (1-100),
                    "estimated_salary": integer (in IDR),
                    "reason": string
                }}
            ],

            "skill_analysis": {{
                "matched_skills": [
                    {{
                        "skill": string,
                        "score": integer (1-100)
                    }}
                ],

                "missing_skills": [
                    string
                ],

                "market_demand_skills": [
                    {{
                        "skill": string,
                        "demand_score": integer
                    }}
                ]
            }},

            "recommendations": [
                {{
                    "priority": string,
                    "category": string,
                    "title": string,
                    "description": string
                }}
            ]
        }}
    """)

    chain = prompt | chat_model
    response = chain.invoke({
        "cv_profile": cv_profile,
        "job_listings": jobs_text,
        "value_to_extract": value_to_extract
    })

    response_json = "".join(
                part["text"]
                for part in response.content
                if part["type"] == "text"
            )
    
    data = json.loads(response_json)

    with open("output.json", "w") as file:
        json.dump(data, file, indent=4)

In [ ]:
# Load index  
dynamic_store = Chroma(
    persist_directory="./chroma_db/dynamic_csv",
    embedding_function=embedding_model_hf
)

query = f"""
        Find at least two of the most suitable job based on candidate's skill set for

        Role:
        AI Engineer

        Profile candidate:
        {extract_cv("dataset/CV_Prayoga Rasyid Sudrajat-1-2.pdf")}
    """
# Find matching job
job_matches = dynamic_store.similarity_search(query=query, k=5, filter={"job_category": "AI Engineer"})
jobs_text = "\n\n---\n\n".join([d.page_content for d in job_matches]) # Formatting

for job in job_matches:
    print(job.page_content)

site: indeed
job_title: AI Engineer (Data)
date_posted: 2026-06-23 00:00:00
tech_stack: LLM, RAG, SQL, LangChain, LlamaIndex, BigQuery, Tableau, dbt, Airflow
requirements: Bachelor's, Communication, Professional, English
min_salary: 
max_salary: 
industry: 
job_description: **About The Role**



We are looking for an AI Engineer (Data) to help build and scale AI\-powered solutions that enhance how our teams access, analyze, and utilize data. As part of the Data team, you will develop agentic applications and AI\-driven tools that leverage Large Language Models (LLMs) to improve internal workflows, enable self\-service analytics, and increase the impact of data across the organization.



In addition to building AI solutions, you will work closely with data analysts and engineers to design data architectures, prototype innovative use cases, and transform large\-scale data into actionable insights that support strategic decision\-making.


**What You Will Do**
site: jobstreet
job_title: 

In [11]:
dynamic_store = Chroma(
    persist_directory="./chroma_db/dynamic_csv",
    embedding_function=embedding_model_hf
)
print(dynamic_store._collection.count())

1637


In [9]:
import shutil
import os

persist_dir = "./chroma_db/dynamic_csv"

if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)

job_processing(file_path="dataset/jobs_final.csv", meta_data_cols=["job_category"])

Stored to vector db using Hugging Face.
Dynamic index updated: 1637 chunks


In [15]:
# Check result
result = match_cv_to_jobs(cv_path="dataset/CV_Prayoga Rasyid Sudrajat-1-2.pdf", value_to_extract="Data Scientist")